In [6]:
import pandas as pd
import requests
import urllib
from datetime import datetime

In [16]:
def get_json(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}
    # headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
    r=requests.get(url, headers=headers)
    if r.ok:
        data=r.json()
        if data['returncode']!=200:
            data=None
    else:
        data=None
    return data 

In [8]:
def get_df(url):
    data=get_json(url)
    if data is not None:
        name={i['code']: i['cname'] for i in data['returndata']['wdnodes'][0]['nodes']}
        value=[i['data']['data']  for i in data['returndata']['datanodes']]
        date=[i['wds'][1]['valuecode']  for i in data['returndata']['datanodes']]
        code=[i['wds'][0]['valuecode']  for i in data['returndata']['datanodes']]

        df=pd.DataFrame(dict(zip(['Code', 'Date', 'Value'], [code, date, value])))
        df['Name']=df['Code'].map(name)
        return df
    else:
        return None

In [9]:
def get_url(start_time, end_time, sector):
   # get current timestamp
   now = datetime.now()
   timestamp = int(now.timestamp())*1000

   # generate enquiry dict
   time=f'{start_time}-{end_time}'
   
   dict_time={'wdcode': 'sj', 'valuecode': time}
   dict_sector={'wdcode': 'zb', 'valuecode': sector}
   str_dfwds=(f'{dict_sector},{dict_time}').replace(' ','').replace('\'','\"')

   # formulate url
   f={'m':'QueryData', 'dbcode':'hgyd', 'rowcode':'zb', 'colcode':'sj', 'wds':'[]', 
      'dfwds':f'[{str_dfwds}]', 'k1':f'{timestamp}'}
   main_url='https://data.stats.gov.cn/easyquery.htm'
   url=f'{main_url}?{urllib.parse.urlencode(f)}'
   return url


start_time='202304'
end_time='202305'
sector='A0601' #房地產
sector='A030102' #原油产量
sector='A0E01' #失業率
sector='A0801' #進出口
url=get_url(start_time, end_time, sector)

In [10]:
url


'https://data.stats.gov.cn/easyquery.htm?m=QueryData&dbcode=hgyd&rowcode=zb&colcode=sj&wds=%5B%5D&dfwds=%5B%7B%22wdcode%22%3A%22zb%22%2C%22valuecode%22%3A%22A0801%22%7D%2C%7B%22wdcode%22%3A%22sj%22%2C%22valuecode%22%3A%22202304-202305%22%7D%5D&k1=1721298398000'

In [11]:

def get_data_content_list(code):
    start_time='202304'
    end_time='202305'
    url=get_url(start_time, end_time, code)
    df=get_df(url)
    if df is None:
        lst=None
    else:
        lst=df['Name'].drop_duplicates().to_list()
    return {code: lst}



In [12]:
lst_code=[f'A0{str(i)}{str(j).zfill(2)}' for i in range(1,11) for j in range(0,11)]
lst_code[:10]


['A0100',
 'A0101',
 'A0102',
 'A0103',
 'A0104',
 'A0105',
 'A0106',
 'A0107',
 'A0108',
 'A0109']

In [17]:
from time import sleep
dict_content={}
for code in lst_code[:50]:
    dict_content.update(get_data_content_list(code))
    sleep(1)

In [15]:
url

'https://data.stats.gov.cn/easyquery.htm?m=QueryData&dbcode=hgyd&rowcode=zb&colcode=sj&wds=%5B%5D&dfwds=%5B%7B%22wdcode%22%3A%22zb%22%2C%22valuecode%22%3A%22A0801%22%7D%2C%7B%22wdcode%22%3A%22sj%22%2C%22valuecode%22%3A%22202304-202305%22%7D%5D&k1=1721298398000'

In [20]:
dict_content

{'A0100': None,
 'A0101': None,
 'A0102': None,
 'A0103': None,
 'A0104': None,
 'A0105': None,
 'A0106': None,
 'A0107': None,
 'A0108': None,
 'A0109': ['工业生产者出厂价格指数(上年同月=100)',
  '煤炭开采和洗选业工业生产者出厂价格指数(上年同月=100)',
  '石油和天然气开采业工业生产者出厂价格指数(上年同月=100)',
  '黑色金属矿采选业工业生产者出厂价格指数(上年同月=100)',
  '有色金属矿采选业工业生产者出厂价格指数(上年同月=100)',
  '非金属矿采选业工业生产者出厂价格指数(上年同月=100)',
  '其他采矿业工业生产者出厂价格指数(上年同月=100)',
  '农副食品加工业工业生产者出厂价格指数(上年同月=100)',
  '食品制造业工业生产者出厂价格指数(上年同月=100)',
  '饮料制造业工业生产者出厂价格指数(上年同月=100)',
  '烟草制品业工业生产者出厂价格指数(上年同月=100)',
  '纺织业工业生产者出厂价格指数(上年同月=100)',
  '纺织服装、鞋、帽制造业工业生产者出厂价格指数(上年同月=100)',
  '皮革、毛皮、羽毛(绒)及其制品业工业生产者出厂价格指数(上年同月=100)',
  '木材加工及木、竹、藤、棕、草制品业工业生产者出厂价格指数(上年同月=100)',
  '家具制造业工业生产者出厂价格指数(上年同月=100)',
  '造纸及纸制品业工业生产者出厂价格指数(上年同月=100)',
  '印刷业和记录媒介的复制工业生产者出厂价格指数(上年同月=100)',
  '文教体育用品制造业工业生产者出厂价格指数(上年同月=100)',
  '石油加工、炼焦及核燃料加工业工业生产者出厂价格指数(上年同月=100)',
  '化学原料及化学制品制造业工业生产者出厂价格指数(上年同月=100)',
  '医药制造业工业生产者出厂价格指数(上年同月=100)',
  '化学纤维制造业工业生产者出厂价格指数(上年同月=100)',
  '橡胶制品业工业生产者出厂价格指数(上年同月=100)',
  '塑料制品业工业

In [ ]:
# df['Date']=pd.to_datetime(df['Date'], format='%Y%m')
# df[df['Name'].str.contains('进出口差额_累计值')][['Date', 'Value']].set_index('Date').plot(title='Average working hour')